<a href="https://colab.research.google.com/github/muzzamilmehmood28-ops/Heart_disease_prediction/blob/main/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd

# Load your dataset (replace 'your_dataset.csv' with your actual file name)
df = pd.read_csv('dataset.csv')

# Display the first few rows
print(df.head())

# Check for missing values in each column
print(df.isnull().sum())
print("Dataset loaded successfully!")

   age     sex   bmi     region income_level smoking_status alcohol_intake  \
0   57    Male  27.1  Northeast       Middle          Never       Moderate   
1   40  Female  29.0  Northeast          Low          Never    Non-drinker   
2   63  Female  40.1  Southeast         High          Never       Moderate   
3   66  Female  35.5       West       Middle          Never          Heavy   
4   29  Female  19.8  Northeast         High        Current    Non-drinker   

  physical_activity  diet_quality  sleep_hours  ...  cholesterol  \
0             Light             7          5.7  ...          204   
1         Sedentary             1          8.0  ...          173   
2         Sedentary             6          5.5  ...          209   
3         Sedentary            10          7.4  ...          187   
4         Sedentary             8          8.3  ...          247   

   fasting_blood_sugar  max_heart_rate       resting_ecg  exercise_angina  \
0                    0             195       

In [7]:
# Fill missing values for numerical columns with the median
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill missing values for categorical columns with the mode (first value)
cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Verify that no missing values remain
print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0


/tmp/ipykernel_699/4283392103.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [8]:
# Check initial shape of the dataset
print("Shape before removing duplicates:", df.shape)

# Drop duplicate rows
df = df.drop_duplicates()

# Check shape after removing duplicates
print("Shape after removing duplicates:", df.shape)

Shape before removing duplicates: (100000, 26)
Shape after removing duplicates: (100000, 26)


In [9]:
from sklearn.preprocessing import LabelEncoder

# Initialize Label Encoder
label_encoder = LabelEncoder()

# Identify categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# Apply Label Encoding to categorical columns
for col in cat_cols:
    df[col] = label_encoder.fit_transform(df[col].astype(str))

print("Categorical columns successfully encoded!")
print(df.head())

Categorical columns successfully encoded!
   age  sex   bmi  region  income_level  smoking_status  alcohol_intake  \
0   57    1  27.1       1             2               2               1   
1   40    0  29.0       1             1               2               2   
2   63    0  40.1       2             0               2               1   
3   66    0  35.5       4             2               2               0   
4   29    0  19.8       1             0               0               2   

   physical_activity  diet_quality  sleep_hours  ...  cholesterol  \
0                  1             7          5.7  ...          204   
1                  2             1          8.0  ...          173   
2                  2             6          5.5  ...          209   
3                  2            10          7.4  ...          187   
4                  2             8          8.3  ...          247   

   fasting_blood_sugar  max_heart_rate  resting_ecg  exercise_angina  \
0                   

In [10]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target variable (y)
# Replace 'heart_disease' with your actual target column name if it's different
target_column = 'heart_disease'

X = df.drop(columns=[target_column])
y = df[target_column]

# Scale numerical columns
scaler = StandardScaler()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("Numerical features successfully scaled!")
print(X.head())

Numerical features successfully scaled!
        age       sex       bmi    region  income_level  smoking_status  \
0  0.281332  0.901322 -0.176397 -0.652074      0.984348        0.716095   
1 -1.084262 -1.109481  0.171432 -0.652074     -0.248537        0.716095   
2  0.763306 -1.109481  2.203484  0.066671     -1.481423        0.716095   
3  1.004293 -1.109481  1.361372  1.504160      0.984348        0.716095   
4 -1.967882 -1.109481 -1.512792 -0.652074     -1.481423       -2.035788   

   alcohol_intake  physical_activity  diet_quality  sleep_hours  ...  \
0       -0.350343          -0.389628      0.584135    -1.107703  ...   
1        1.119587           0.572346     -2.396829     0.999161  ...   
2       -0.350343           0.572346      0.087307    -1.290909  ...   
3       -1.820273           0.572346      2.074617     0.449545  ...   
4        1.119587           0.572346      1.080962     1.273970  ...   

   resting_bp  cholesterol  fasting_blood_sugar  max_heart_rate  resting_ecg

In [11]:
from sklearn.model_selection import train_test_split

# Split into 80% training and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Testing set shape: X_test = {X_test.shape}, y_test = {y_test.shape}")

Training set shape: X_train = (80000, 25), y_train = (80000,)
Testing set shape: X_test = (20000, 25), y_test = (20000,)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import time

# 1. Logistic Regression
print("Training Logistic Regression...")
start_time = time.time()
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)
lr_train_time = time.time() - start_time

print("\n model successfully trained!")

Training Logistic Regression...

 model successfully trained!


In [14]:
# 2. Decision Tree Classifier
print("Training Decision Tree...")
start_time = time.time()
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_train_time = time.time() - start_time
print("\n model successfully trained!")

Training Decision Tree...

 model successfully trained!


In [16]:
# 3. Random Forest Classifier
print("Training Random Forest...")
start_time = time.time()
rf_model = RandomForestClassifier(random_state=42, n_estimators=50)
rf_model.fit(X_train, y_train)
rf_train_time = time.time() - start_time
print("\n model successfully trained!")

Training Random Forest...

 model successfully trained!


In [19]:
# 4. K-Nearest Neighbors (KNN)
# Note: On 100k rows, KNN can take a moment. We'll use n_neighbors=5.
print("Training KNN...")
start_time = time.time()
knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1) # n_jobs=-1 uses all CPU cores to speed it up
knn_model.fit(X_train, y_train)
knn_train_time = time.time() - start_time

print("\n model successfully trained!")

Training KNN...

 model successfully trained!


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Create a dictionary to store models for easy evaluation loop
models = {
    "Logistic Regression": lr_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model,
    "KNN": knn_model
}

evaluation_results = {}

for name, model in models.items():
    print(f"\n Evaluating {name} ")

    # Make predictions on the test set
    y_pred = model.predict(X_test)

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)

    # Store results
    evaluation_results[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "Confusion Matrix": cm
    }

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("Confusion Matrix:\n", cm)


 Evaluating Logistic Regression 
Accuracy:  0.7688
Precision: 0.7684
Recall:    0.7688
F1 Score:  0.7683
Confusion Matrix:
 [[8714 2111]
 [2514 6661]]

 Evaluating Decision Tree 
Accuracy:  0.6667
Precision: 0.6670
Recall:    0.6667
F1 Score:  0.6668
Confusion Matrix:
 [[7445 3380]
 [3286 5889]]

 Evaluating Random Forest 
Accuracy:  0.7577
Precision: 0.7574
Recall:    0.7577
F1 Score:  0.7569
Confusion Matrix:
 [[8706 2119]
 [2727 6448]]

 Evaluating KNN 
Accuracy:  0.7231
Precision: 0.7225
Recall:    0.7231
F1 Score:  0.7221
Confusion Matrix:
 [[8368 2457]
 [3081 6094]]
